In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from collections.abc import Callable

import logging
from dataclasses import dataclass
import datetime

In [2]:
%%capture

# Make sure to add project root directory to PYTHONPATH
# export PYTHONPATH="${PYTHONPATH}:${pwd}"

# Change directory to project root directory

%cd ..

In [3]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [4]:
dtypes = {
    "value": np.float32,
    "label": np.int32,
}

In [5]:
data_path = Path("data")

raw_data_path = Path("data") / "raw" / "yahoo"

In [6]:
a1_benchmark_path = raw_data_path / "A1Benchmark"
a2_benchmark_path = raw_data_path / "A2Benchmark"
a3_benchmark_path = raw_data_path / "A3Benchmark"
a4_benchmark_path = raw_data_path / "A4Benchmark"

In [7]:
def extract_number(file_path: Path) -> int:
    return int(file_path.stem.split('_')[-1])


def extract_t_number(file_path: Path) -> int:
    return int(file_path.stem.split('-TS')[-1])


a1_filenames = sorted([filename for filename in sorted(a1_benchmark_path.glob("*.csv"))], key=extract_number)
a2_filenames = sorted([filename for filename in sorted(a2_benchmark_path.glob("*.csv"))], key=extract_number)
a3_filenames = sorted([filename for filename in sorted(a3_benchmark_path.glob("*.csv"))], key=extract_t_number)
a4_filenames = sorted([filename for filename in sorted(a4_benchmark_path.glob("*.csv"))], key=extract_t_number)

In [8]:
@dataclass
class A1ServiceUnderMonitoring:
    name: str = "TS"
    tsid: int = 0

    def __repr__(self) -> str:
        return f"{self.name}_{self.tsid}"

In [9]:
def rename_column_names(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    df.columns = ["timestamp", f"{service}_VALUE", f"{service}_IS_ANOMALY"]

    return df


def adjust_data_types(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    df["timestamp"] = df[f"timestamp"].astype(dtypes["label"])
    df[f"{service}_VALUE"] = df[f"{service}_VALUE"].astype(dtypes["value"])
    df[f"{service}_IS_ANOMALY"] = df[f"{service}_IS_ANOMALY"].astype(dtypes["label"])

    return df


def set_index(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    df.index = df["timestamp"]
    df = df.drop(columns=["timestamp"])

    return df


def remove_duplicates(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    df = df[~df.index.duplicated(keep="first")]

    return df

## Yahoo A1 Benchmark

In [10]:
a1_common_transformations: list[Callable] = [
    rename_column_names,
    adjust_data_types,
    set_index,
    remove_duplicates,
]


def apply_a1_common_transformations(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    for common_t in a1_common_transformations:
        df = common_t(df, *args, **kwargs)

    return df


def transform_a1_raw_data(df: pd.DataFrame, service: A1ServiceUnderMonitoring) -> pd.DataFrame:
    df = apply_a1_common_transformations(df, service)

    return df

In [11]:
df_a1 = pd.DataFrame()

for i, filename in enumerate(a1_filenames):
    tsid = extract_number(filename)
    service = A1ServiceUnderMonitoring(tsid=tsid)

    _df = pd.read_csv(filename)

    df_transformed = transform_a1_raw_data(_df.copy(), service)

    df_a1 = df_a1.join(df_transformed, how="outer")

In [12]:
df_a1 = df_a1.fillna(0)

In [13]:
df_a1.head()

,TS_1_VALUE,TS_1_IS_ANOMALY,TS_2_VALUE,TS_2_IS_ANOMALY,TS_3_VALUE,TS_3_IS_ANOMALY,TS_4_VALUE,TS_4_IS_ANOMALY,TS_5_VALUE,TS_5_IS_ANOMALY,...,TS_63_VALUE,TS_63_IS_ANOMALY,TS_64_VALUE,TS_64_IS_ANOMALY,TS_65_VALUE,TS_65_IS_ANOMALY,TS_66_VALUE,TS_66_IS_ANOMALY,TS_67_VALUE,TS_67_IS_ANOMALY
timestamp,,,,,,,,,,,,,,,,,,,,,
1,0.000000,0.0,12183.0,0.0,3.716667,0,5.0,0.0,2109.0,0.0,...,13.678001,0.0,5.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0
2,0.091758,0.0,12715.0,0.0,3.610833,0,60.0,0.0,3229.0,0.0,...,10.897359,0.0,7.0,0.0,434.0,0.0,3.0,0.0,48.0,0.0
3,0.172297,0.0,12736.0,0.0,3.481389,0,88.0,0.0,3637.0,0.0,...,12.525914,0.0,4.0,0.0,310.0,0.0,8.0,0.0,55.0,0.0
4,0.226219,0.0,12716.0,0.0,3.380278,0,84.0,0.0,1982.0,0.0,...,12.335350,0.0,5.0,0.0,201.0,0.0,32.0,0.0,122.0,0.0
5,0.176358,0.0,12739.0,0.0,3.193333,0,111.0,0.0,2751.0,0.0,...,15.313461,0.0,0.0,0.0,221.0,0.0,95.0,0.0,92.0,0.0


In [15]:
df_a1.to_parquet("data/consolidated/yahoo_a1.parquet")

## Yahoo A2 Benchmark

In [14]:
@dataclass
class A2ServiceUnderMonitoring:
    name: str = "TS"
    tsid: int = 0

    def __repr__(self) -> str:
        return f"{self.name}_{self.tsid}"

In [15]:
def adjust_a2_data_types(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    df["timestamp"] = df["timestamp"].apply(lambda x: datetime.datetime.fromtimestamp(x))

    
    df[f"{service}_VALUE"] = df[f"{service}_VALUE"].astype(dtypes["value"])
    df[f"{service}_IS_ANOMALY"] = df[f"{service}_IS_ANOMALY"].astype(dtypes["label"])

    return df


a2_common_transformations: list[Callable] = [
    rename_column_names,
    adjust_a2_data_types,
    set_index,
    remove_duplicates,
]


def apply_a2_common_transformations(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    for common_t in a2_common_transformations:
        df = common_t(df, *args, **kwargs)

    return df


def transform_a2_raw_data(df: pd.DataFrame, service: A2ServiceUnderMonitoring) -> pd.DataFrame:
    df = apply_a2_common_transformations(df, service)

    return df

In [16]:
df_a2 = pd.DataFrame()

for i, filename in enumerate(a2_filenames):
    tsid = extract_number(filename)
    service = A2ServiceUnderMonitoring(tsid=tsid)

    _df = pd.read_csv(filename)

    df_transformed = transform_a2_raw_data(_df.copy(), service)

    df_a2 = df_a2.join(df_transformed, how="outer")

In [17]:
df_a2.head()

,TS_1_VALUE,TS_1_IS_ANOMALY,TS_2_VALUE,TS_2_IS_ANOMALY,TS_3_VALUE,TS_3_IS_ANOMALY,TS_4_VALUE,TS_4_IS_ANOMALY,TS_5_VALUE,TS_5_IS_ANOMALY,...,TS_96_VALUE,TS_96_IS_ANOMALY,TS_97_VALUE,TS_97_IS_ANOMALY,TS_98_VALUE,TS_98_IS_ANOMALY,TS_99_VALUE,TS_99_IS_ANOMALY,TS_100_VALUE,TS_100_IS_ANOMALY
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-11-23 08:00:00,13.894032,0,27.109888,0,-19.413647,0,52.606331,0,64.446228,0,...,-223.756699,0,479.531036,0,524.467224,0,-242.270447,0,530.537598,0
2014-11-23 09:00:00,33.578274,0,40.776443,0,61.443260,0,53.295685,0,57.734577,0,...,145.382828,0,-15.503972,0,-9.857374,0,114.314301,0,-36.182106,0
2014-11-23 10:00:00,88.933746,0,122.541466,0,85.501343,0,186.898743,0,216.319717,0,...,-146.089294,0,693.139587,0,692.235901,0,-244.410446,0,698.380371,0
2014-11-23 11:00:00,125.389427,0,168.222794,0,216.338425,0,249.971954,0,287.103058,0,...,807.535767,0,765.515564,0,741.500732,0,745.833313,0,745.766174,0
2014-11-23 12:00:00,152.962006,0,196.942108,0,232.313782,0,279.808258,0,316.439178,0,...,426.032562,0,540.046387,0,485.773132,0,290.595795,0,472.031189,0


In [24]:
df_a2.isna().any(axis=0).sum()

np.int64(0)

In [25]:
df_a2.isna().any(axis=1).sum()

np.int64(0)

In [26]:
df_a2.to_parquet("data/consolidated/yahoo_a2.parquet")

## Yahoo A3 Benchmark

In [27]:
@dataclass
class A3ServiceUnderMonitoring:
    name: str = "TS"
    tsid: int = 0

    def __repr__(self) -> str:
        return f"{self.name}_{self.tsid}"

In [28]:
def rename_a3_column_names(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    df.columns = [
        "timestamp",
        f"{service}_VALUE",
        f"{service}_IS_ANOMALY",
        f"{service}_CHANGEPOINT",
        f"{service}_ADDITIVE_TREND",
        f"{service}_ADDITIVE_NOISE",
        f"{service}_12HOUR_SEASONALITY",
        f"{service}_DAILY_SEASONALITY",
        f"{service}_WEEKLY_SEASONALITY"
    ]

    return df


def adjust_a3_data_types(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    df["timestamp"] = df["timestamp"].apply(lambda x: datetime.datetime.fromtimestamp(x))

    
    df[f"{service}_VALUE"] = df[f"{service}_VALUE"].astype(dtypes["value"])
    df[f"{service}_IS_ANOMALY"] = df[f"{service}_IS_ANOMALY"].astype(dtypes["label"])

    return df


a3_common_transformations: list[Callable] = [
    rename_a3_column_names,
    adjust_a3_data_types,
    set_index,
    remove_duplicates,
]


def apply_a3_common_transformations(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    for common_t in a3_common_transformations:
        df = common_t(df, *args, **kwargs)

    return df


def transform_a3_raw_data(df: pd.DataFrame, service: A3ServiceUnderMonitoring) -> pd.DataFrame:
    df = apply_a3_common_transformations(df, service)

    return df

In [29]:
df_a3 = pd.DataFrame()

for i, filename in enumerate(a3_filenames):
    tsid = extract_t_number(filename)
    service = A3ServiceUnderMonitoring(tsid=tsid)

    _df = pd.read_csv(filename)

    df_transformed = transform_a3_raw_data(_df.copy(), service)

    df_a3 = df_a3.join(df_transformed, how="outer")

In [30]:
df_a3.isnull().values.any()

np.False_

In [31]:
df_a3.head()

,TS_1_VALUE,TS_1_IS_ANOMALY,TS_1_CHANGEPOINT,TS_1_ADDITIVE_TREND,TS_1_ADDITIVE_NOISE,TS_1_12HOUR_SEASONALITY,TS_1_DAILY_SEASONALITY,TS_1_WEEKLY_SEASONALITY,TS_2_VALUE,TS_2_IS_ANOMALY,...,TS_99_DAILY_SEASONALITY,TS_99_WEEKLY_SEASONALITY,TS_100_VALUE,TS_100_IS_ANOMALY,TS_100_CHANGEPOINT,TS_100_ADDITIVE_TREND,TS_100_ADDITIVE_NOISE,TS_100_12HOUR_SEASONALITY,TS_100_DAILY_SEASONALITY,TS_100_WEEKLY_SEASONALITY
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-11-23 07:00:00,-363.278900,0,0,-2,-361.278909,0.000000,0.000000,0.000000,-46.394356,0,...,0.000000,0.000000,6.470061,0,0,0,6.470061,0.000000,0.000000,0.000000
2014-11-23 08:00:00,320.888580,0,0,-4,-217.845824,413.100000,115.769759,13.864655,311.346222,0,...,24.639573,3.350251,80.972290,0,0,0,-11.077853,67.600000,22.206674,2.243472
2014-11-23 09:00:00,891.727417,0,0,-6,-69.142686,715.510189,223.650000,27.709919,543.279053,0,...,47.600000,6.695816,159.014679,0,0,0,-5.455761,117.086635,42.900000,4.483806
2014-11-23 10:00:00,1174.652344,0,0,-8,-1.353004,826.200000,316.288863,41.516428,603.441956,0,...,67.316566,10.032017,214.721802,0,0,0,12.134172,135.200000,60.669762,6.717869
2014-11-23 11:00:00,1712.290283,0,0,-10,564.142037,715.510189,387.373163,55.264872,652.807251,0,...,82.445618,13.354187,201.695831,0,0,0,1.361684,117.086635,74.304980,8.942536


In [34]:
df_a3[[col for col in df_a3.columns if col.endswith("VALUE") or col.endswith("IS_ANOMALY")]]

,TS_1_VALUE,TS_1_IS_ANOMALY,TS_2_VALUE,TS_2_IS_ANOMALY,TS_3_VALUE,TS_3_IS_ANOMALY,TS_4_VALUE,TS_4_IS_ANOMALY,TS_5_VALUE,TS_5_IS_ANOMALY,...,TS_96_VALUE,TS_96_IS_ANOMALY,TS_97_VALUE,TS_97_IS_ANOMALY,TS_98_VALUE,TS_98_IS_ANOMALY,TS_99_VALUE,TS_99_IS_ANOMALY,TS_100_VALUE,TS_100_IS_ANOMALY
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-11-23 07:00:00,-363.278900,0,-46.394356,0,14.576078,0,-6.129439,0,-12.491440,0,...,-36.623085,0,-2.763855,0,-18.698503,0,-1.097379,0,6.470061,0
2014-11-23 08:00:00,320.888580,0,311.346222,0,265.986694,0,251.488632,0,389.460022,0,...,336.452942,0,254.258453,0,430.393677,0,107.517242,0,80.972290,0
2014-11-23 09:00:00,891.727417,0,543.279053,0,427.191956,0,645.865417,0,632.545715,0,...,831.443420,0,479.288269,0,915.535400,0,196.721268,0,159.014679,0
2014-11-23 10:00:00,1174.652344,0,603.441956,0,547.133850,0,834.902344,0,729.666260,0,...,963.322876,0,578.198608,0,1049.605225,0,242.941574,0,214.721802,0
2014-11-23 11:00:00,1712.290283,0,652.807251,0,528.638855,0,891.027344,0,724.301453,0,...,1086.901855,0,596.264160,0,1036.215576,0,236.049438,0,201.695831,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-02-01 02:00:00,-4232.315430,0,4620.315918,0,2957.535889,0,2594.749268,0,2777.278564,0,...,-6018.449707,0,-3834.324219,0,-4120.329102,0,-5223.139648,0,-143.374161,0
2015-02-01 03:00:00,-4343.327148,0,4396.719238,0,2845.511230,0,2542.612061,0,2582.842773,0,...,-6058.724121,0,-3935.881104,0,-4544.775391,0,-5279.851074,0,-213.830734,0
2015-02-01 04:00:00,-4415.659180,0,4424.091309,0,2819.062256,0,2441.076172,0,2566.612549,0,...,-6072.104980,0,-3949.737305,0,-4401.919434,0,-5290.376465,0,-191.906219,0


In [35]:
df_a3 = df_a3[[col for col in df_a3.columns if col.endswith("VALUE") or col.endswith("IS_ANOMALY")]]

In [39]:
df_a3.isna().any(axis=0).sum()

np.int64(0)

In [40]:
df_a3.isna().any(axis=1).sum()

np.int64(0)

In [41]:
df_a3.to_parquet("data/consolidated/yahoo_a3.parquet")

## Yahoo A4 Benchmark

In [42]:
@dataclass
class A4ServiceUnderMonitoring:
    name: str = "TS"
    tsid: int = 0

    def __repr__(self) -> str:
        return f"{self.name}_{self.tsid}"

In [43]:
def rename_a4_column_names(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    df.columns = [
        "timestamp",
        f"{service}_VALUE",
        f"{service}_IS_ANOMALY",
        f"{service}_CHANGEPOINT",
        f"{service}_ADDITIVE_TREND",
        f"{service}_ADDITIVE_NOISE",
        f"{service}_12HOUR_SEASONALITY",
        f"{service}_DAILY_SEASONALITY",
        f"{service}_WEEKLY_SEASONALITY"
    ]

    return df


def adjust_a4_data_types(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    df["timestamp"] = df["timestamp"].apply(lambda x: datetime.datetime.fromtimestamp(x))

    
    df[f"{service}_VALUE"] = df[f"{service}_VALUE"].astype(dtypes["value"])
    df[f"{service}_IS_ANOMALY"] = df[f"{service}_IS_ANOMALY"].astype(dtypes["label"])

    return df


a4_common_transformations: list[Callable] = [
    rename_a4_column_names,
    adjust_a4_data_types,
    set_index,
    remove_duplicates,
]


def apply_a4_common_transformations(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    for common_t in a4_common_transformations:
        df = common_t(df, *args, **kwargs)

    return df


def transform_a4_raw_data(df: pd.DataFrame, service: A4ServiceUnderMonitoring) -> pd.DataFrame:
    df = apply_a4_common_transformations(df, service)

    return df

In [44]:
df_a4 = pd.DataFrame()

for i, filename in enumerate(a4_filenames):
    tsid = extract_t_number(filename)
    service = A4ServiceUnderMonitoring(tsid=tsid)

    _df = pd.read_csv(filename)

    df_transformed = transform_a4_raw_data(_df.copy(), service)

    df_a4 = df_a4.join(df_transformed, how="outer")

In [45]:
df_a4.isnull().values.any()

np.False_

In [46]:
df_a4.head()

,TS_1_VALUE,TS_1_IS_ANOMALY,TS_1_CHANGEPOINT,TS_1_ADDITIVE_TREND,TS_1_ADDITIVE_NOISE,TS_1_12HOUR_SEASONALITY,TS_1_DAILY_SEASONALITY,TS_1_WEEKLY_SEASONALITY,TS_2_VALUE,TS_2_IS_ANOMALY,...,TS_99_DAILY_SEASONALITY,TS_99_WEEKLY_SEASONALITY,TS_100_VALUE,TS_100_IS_ANOMALY,TS_100_CHANGEPOINT,TS_100_ADDITIVE_TREND,TS_100_ADDITIVE_NOISE,TS_100_12HOUR_SEASONALITY,TS_100_DAILY_SEASONALITY,TS_100_WEEKLY_SEASONALITY
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-11-23 07:00:00,2.889225,0,0,3,-0.110775,0.000000,0.000000,0.000000,4.946131,0,...,0.000000,0.000000,-10.258903,0,0,0,-10.258903,0.000000,0.000000,0.000000
2014-11-23 08:00:00,112.492218,0,0,6,46.188395,45.900000,12.863307,1.540517,177.315567,0,...,16.926766,1.529300,141.132614,0,0,0,-5.156569,94.200000,46.121554,5.967635
2014-11-23 09:00:00,131.097061,0,0,9,14.667044,79.501132,24.850000,3.078880,276.682465,0,...,32.700000,3.056461,260.483276,0,0,0,-3.702823,163.159186,89.100000,11.926923
2014-11-23 10:00:00,128.208832,0,0,12,-15.347308,91.800000,35.143207,4.612936,351.276215,0,...,46.244783,4.579347,319.502991,0,0,0,-12.772982,188.400000,126.006428,17.869530
2014-11-23 11:00:00,121.639030,0,0,15,-22.044107,79.501132,43.041463,6.140541,337.027985,0,...,56.638061,6.095829,346.294525,0,0,0,5.022469,163.159186,154.325727,23.787146


In [47]:
df_a4[[col for col in df_a4.columns if col.endswith("VALUE") or col.endswith("IS_ANOMALY")]]

,TS_1_VALUE,TS_1_IS_ANOMALY,TS_2_VALUE,TS_2_IS_ANOMALY,TS_3_VALUE,TS_3_IS_ANOMALY,TS_4_VALUE,TS_4_IS_ANOMALY,TS_5_VALUE,TS_5_IS_ANOMALY,...,TS_96_VALUE,TS_96_IS_ANOMALY,TS_97_VALUE,TS_97_IS_ANOMALY,TS_98_VALUE,TS_98_IS_ANOMALY,TS_99_VALUE,TS_99_IS_ANOMALY,TS_100_VALUE,TS_100_IS_ANOMALY
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-11-23 07:00:00,2.889225,0,4.946131,0,16.049948,0,-6.129439,0,-104.190781,0,...,-20.883760,0,134.126740,0,17.647295,0,-6.165484,0,-10.258903,0
2014-11-23 08:00:00,112.492218,0,177.315567,0,336.599213,0,251.488632,0,533.741821,0,...,509.367340,0,281.787842,0,247.829376,0,76.609467,0,141.132614,0
2014-11-23 09:00:00,131.097061,0,276.682465,0,547.794739,0,645.865417,0,1023.166687,0,...,790.261047,0,432.315582,0,425.722534,0,119.472794,0,260.483276,0
2014-11-23 10:00:00,128.208832,0,351.276215,0,706.897827,0,834.902344,0,1118.788452,0,...,1034.526855,0,632.952759,0,500.590942,0,112.542381,0,319.502991,0
2014-11-23 11:00:00,121.639030,0,337.027985,0,696.496338,0,891.027344,0,1154.433105,0,...,1087.035889,0,547.288757,0,614.080994,0,181.057281,0,346.294525,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-02-01 02:00:00,1970.317505,0,4753.368164,0,-2741.801514,0,2594.749268,0,-2246.491943,0,...,1786.033447,0,-2739.352051,0,4341.803223,0,115.926613,0,-1725.734375,0
2015-02-01 03:00:00,1889.385864,0,4699.813965,0,-2834.249756,0,2542.612061,0,-2266.349854,0,...,1850.870361,0,-2814.440918,0,4251.833008,0,4.145356,0,-1786.029907,0
2015-02-01 04:00:00,1909.651611,0,4712.094727,0,-2850.192627,0,2441.076172,0,-2279.892090,0,...,1778.271362,0,-2816.079834,0,4292.669922,0,-40.038593,0,-1782.128540,0


In [48]:
df_a4 = df_a4[[col for col in df_a4.columns if col.endswith("VALUE") or col.endswith("IS_ANOMALY")]]

In [49]:
df_a4.to_parquet("data/consolidated/yahoo_a4.parquet")